In [237]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics import precision_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from catboost import CatBoostClassifier

In [238]:
seed = 42
root_path = "/home/stefan/ioai-prep/kits/dataset-165ebde5-4481-4311-964f-be3781e9df2b"

# Data

In [239]:
train = pd.read_csv(f"{root_path}/train.csv")
test = pd.read_csv(f"{root_path}/test.csv")

# Subtask 1

In [240]:
def gfr_class(gfr):
    if gfr >= 90:
        return "Normal"
    elif gfr >= 60:
        return "Mildly Decreased"
    return "Decreased"


subtask1 = test["GFR"].apply(gfr_class)

# Subtask 2

In [241]:
sc_q = train["Serum Creatinine"].quantile([0.25, 0.5, 0.75])


def sc_class(val):
    if val <= sc_q[0.25]:
        return "Very Low"
    elif val <= sc_q[0.5]:
        return "Low"
    elif val <= sc_q[0.75]:
        return "High"
    return "Very High"


subtask2 = test["Serum Creatinine"].apply(sc_class)

# Subtask 3

In [242]:
bmi_med = train["BMI"].median()
subtask3 = (test["BMI"] > bmi_med).astype(int)

# Subtask 4

In [243]:
t_stage_counts = train["T Stage"].value_counts()
subtask4 = test["T Stage"].map(t_stage_counts).fillna(0).astype(int)

# Subtask 5

In [244]:
def prep_df(df: pd.DataFrame):
    df = df.copy()
    df.drop(["ID", "Race"], axis=1, inplace=True)    
    num_cols = df.select_dtypes(include=["float64", "int64"]).columns
    df[num_cols] = df[num_cols].fillna(df[num_cols].median())
    return df

In [250]:
X = prep_df(train.drop(["Status"], axis=1))
y = (train["Status"] == "Dead").astype(int)

In [251]:
X.isna().sum().sort_values(ascending=False)

6th Stage                 103
Age                         0
Marital Status              0
T Stage                     0
N Stage                     0
differentiate               0
Grade                       0
A Stage                     0
Tumor Size                  0
Estrogen Status             0
Progesterone Status         0
Regional Node Examined      0
Reginol Node Positive       0
T_N_Stage                   0
Hormone_Status              0
Reginol Node Negative       0
Blood Pressure              0
Diastolic Pressure          0
Cholesterol                 0
Body Temperature            0
Oxygen Saturation           0
Respiratory Rate            0
Blood Glucose               0
BMI                         0
Heart Rate                  0
Serum Creatinine            0
Uric Acid                   0
Hemoglobin                  0
GFR                         0
Serum Sodium                0
Serum Potassium             0
Serum Albumin               0
Lactate                     0
dtype: int

In [252]:
X.drop(["6th Stage"], axis=1, inplace=True)

In [253]:
num_cols = X.select_dtypes(exclude=["object"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ]
)

In [254]:
def evaluate(model):
    cv = cross_val_score(model, X, y, scoring="precision", cv=3, n_jobs=-1)
    return cv.mean().item()

In [255]:
log = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", LogisticRegression(max_iter=1000, class_weight='balanced', random_state=seed))
])

evaluate(log)

0.2747116481131826

In [256]:
svc = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", LinearSVC(C=1.0, max_iter=5000, class_weight='balanced', random_state=seed))
])

evaluate(svc)

0.27384559884559884

In [270]:
rf = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", RandomForestClassifier(n_estimators=1000, random_state=seed, n_jobs=-1, max_depth=3,
                                     class_weight={0: 1, 1: 2}))
])

evaluate(rf)

0.6184210526315789

In [271]:
model = rf
model.fit(X, y)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  Index(['Age', 'Tumor Size', 'Regional Node Examined', 'Reginol Node Positive',
       'Reginol Node Negative', 'Blood Pressure', 'Diastolic Pressure',
       'Cholesterol', 'Body Temperature', 'Oxygen Saturation',
       'Respiratory Rate', 'Blood Glucose', 'BMI', 'Heart Rate',
       'Serum Creatinine', 'Uric Acid', 'Hemoglobin...
       'Serum Potassium', 'Serum Albumin', 'Lactate'],
      dtype='object')),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['Marital Status', 'T Stage', 'N Stage', 'differentiate', 'Grade',
       'A Stage', 'Estrogen Status', 'Progesterone Status', 'T_N_Stage',
       'Hormone_Status'],
      dtype='object'))])),
                ('model',
                 RandomForestClassifier(class_weight={0: 1, 1: 2}, max_depth=3,
                                        n_estimators=1000, n_jobs=-1,
                                        random_state=42))])

In [272]:
df_test = pd.read_csv(f"{root_path}/test.csv")

test_X = prep_df(df_test)
subtask5 = model.predict(test_X)

In [273]:
subtask5 = pd.Series(subtask5).map({1: "Dead", 0: "Alive"})

In [274]:
subtask5

0      Alive
1      Alive
2      Alive
3      Alive
4      Alive
       ...  
800    Alive
801    Alive
802    Alive
803    Alive
804    Alive
Length: 805, dtype: object

# Submission

In [275]:
subtasks = [
    (1, subtask1),
    (2, subtask2),
    (3, subtask3),
    (4, subtask4),
    (5, subtask5),
]

def build_subtask(sid, answer):
    return pd.DataFrame(
        {"subtaskID": sid, "datapointID": df_test["ID"], "answer": answer}
    )

submission = pd.concat([build_subtask(sid, ans) for sid, ans in subtasks])
submission.to_csv(f"{root_path}/submission.csv", index=False)

In [276]:
submission.head()

,subtaskID,datapointID,answer
0,1,3220,Normal
1,1,3221,Normal
2,1,3222,Normal
3,1,3223,Mildly Decreased
4,1,3224,Mildly Decreased
